In [7]:
# MAE 1001: Introduction to Mechanical and Aerospace Engineering
# Dice Bias Experiment (beginner-friendly version)
#
# WHAT THIS PROGRAM DOES
# ----------------------
# 1. Shows a small web page where you type your name and the name of the die.
# 2. You click a button (1-6) for each roll you make.
# 3. Each roll is saved as a new row in a CSV file (like a simple spreadsheet).
# 4. A histogram (bar chart) updates to show how many times each face came up.
#
# HOW TO RUN IT
# -------------
# First install the libraries (do this once, in your terminal):
#     pip install dash pandas plotly
# (If you are in a Jupyter/Colab notebook, put a ! in front: !pip install dash)
#
# Then run this file:
#     python dice_experiment_beginner.py
# and open the address it prints (usually http://127.0.0.1:8050) in your browser.


# ============================================================
# PART 1: IMPORTS  (tools other people wrote that we reuse)
# ============================================================
import os                          # lets us check whether a file exists
from datetime import datetime      # lets us get today's date

import pandas as pd                # pandas = tables of data ("DataFrames")
import plotly.express as px        # plotly = makes the charts
import dash                        # dash = turns Python into a web page
from dash import dcc, html, Input, Output, State, ctx
#   html -> basic page pieces (headings, buttons, text)
#   dcc  -> "interactive" pieces (text boxes, graphs)
#   Input / Output / State -> used to connect the page to our Python function
#   ctx  -> tells us WHICH button was clicked


# ============================================================
# PART 2: SETTINGS  (values we may want to change later)
# ============================================================
MAX_ENTRIES = 50                   # stop accepting rolls after this many
FACE_VALUES = [1, 2, 3, 4, 5, 6]   # the faces on a six-sided die

# This variable remembers how many rolls have been submitted so far.
# It lives outside any function so it keeps its value between button clicks.
entry_count = 0


# ============================================================
# PART 3: HELPER FUNCTIONS  (small jobs, each with one purpose)
# ============================================================
def make_csv_filename(die_name):
    """Build a file name from the die name and today's date.
    Example: die name 'p_4' on Sept 29, 2025 -> 'p4_092925.csv'
    """
    clean_name = die_name.replace("_", "")           # 'p_4' -> 'p4'
    today = datetime.now().strftime("%m%d%y")        # '092925'
    return f"{clean_name}_{today}.csv"


def load_data(die_name):
    """Read this die's CSV file. If it doesn't exist yet, return an empty table."""
    if not die_name:
        return pd.DataFrame()
    filename = make_csv_filename(die_name)
    if os.path.exists(filename):
        return pd.read_csv(filename)
    return pd.DataFrame()


def save_roll(last_name, first_name, die_name, face_value):
    """Add one new roll to this die's CSV file."""
    filename = make_csv_filename(die_name)

    # Load what we already have (or start with an empty table with column names)
    if os.path.exists(filename):
        df = pd.read_csv(filename)
    else:
        df = pd.DataFrame(columns=["Last Name", "First Name", "Die Name", "Face Value"])

    # Make a one-row table for the new roll and stick it on the bottom
    new_row = pd.DataFrame([{
        "Last Name": last_name,
        "First Name": first_name,
        "Die Name": die_name,
        "Face Value": face_value,
    }])
    df = pd.concat([df, new_row], ignore_index=True)

    # Write everything back to the file (index=False -> no extra row numbers)
    df.to_csv(filename, index=False)


def make_empty_figure():
    """A blank chart shown before there is any data."""
    empty_df = pd.DataFrame({"Face Value": []})
    fig = px.histogram(empty_df, x="Face Value", title="No data yet!")
    fig.update_layout(xaxis={"visible": False}, yaxis={"visible": False})
    return fig


def make_histogram(df, die_name):
    """Turn a table of rolls into a histogram."""
    fig = px.histogram(
        df,
        x="Face Value",
        nbins=6,
        title=f"Histogram of Face Values for Die: {die_name}",
        labels={"Face Value": "Face Value (1-6)"},
        category_orders={"Face Value": FACE_VALUES},   # keep bars in order 1..6
    )
    fig.update_traces(marker=dict(line=dict(width=1, color="DarkSlateGrey")))
    fig.update_layout(bargap=0.1)
    return fig


# ============================================================
# PART 4: THE WEB PAGE LAYOUT  (what the user sees)
# ============================================================
app = dash.Dash(__name__)

app.layout = html.Div([
    html.H2("Dice Roll Experiment Entry"),

    # --- Text boxes ---
    html.Label("Last Name"),
    dcc.Input(id="last_name", type="text", value=""),
    html.Br(),

    html.Label("First Name"),
    dcc.Input(id="first_name", type="text", value=""),
    html.Br(),

    html.Label("Die Name (e.g., o_2 for orange die #2)"),
    dcc.Input(id="die_name", type="text", value=""),
    html.Br(),

    # --- Buttons 1-6 ---
    # This line is a "list comprehension": it builds six buttons in one line.
    # It does the same thing as writing html.Button(...) six times by hand.
    html.Label("Die Face Value"),
    html.Div(
        [html.Button(str(n), id=f"btn_{n}", n_clicks=0) for n in FACE_VALUES],
        style={"display": "flex", "gap": "10px"},
    ),

    # --- Message areas (empty until the program has something to say) ---
    html.Div(id="error_msg", style={"color": "red"}),
    html.Div(id="status_msg", style={"color": "green"}),
    html.Div(id="terminate_msg", style={"color": "blue"}),

    # --- Chart ---
    html.H3("Histogram of Rolls for This Die"),
    dcc.Graph(id="histogram_plot"),
])


# ============================================================
# PART 5: THE CALLBACK  (what happens when a button is clicked)
# ============================================================
# A "callback" connects the page to a Python function:
#   Input  = things that TRIGGER the function (the 6 buttons)
#   State  = things the function READS but that don't trigger it (the text boxes)
#   Output = places on the page where the function's results get displayed
@app.callback(
    [Output("histogram_plot", "figure"),
     Output("error_msg", "children"),
     Output("status_msg", "children"),
     Output("terminate_msg", "children")],
    [Input("btn_1", "n_clicks"),
     Input("btn_2", "n_clicks"),
     Input("btn_3", "n_clicks"),
     Input("btn_4", "n_clicks"),
     Input("btn_5", "n_clicks"),
     Input("btn_6", "n_clicks")],
    [State("last_name", "value"),
     State("first_name", "value"),
     State("die_name", "value")],
)
def update_output(btn_1, btn_2, btn_3, btn_4, btn_5, btn_6,
                  last_name, first_name, die_name):
    global entry_count   # we want to CHANGE the variable defined in Part 2

    error = ""
    status = ""
    terminate = ""

    # --- Step 1: Which button was clicked? ---
    # ctx.triggered_id is the id of the button that fired this function,
    # for example "btn_4". It is None when the page first loads.
    if ctx.triggered_id is None:
        return make_empty_figure(), error, status, terminate

    # "btn_4" -> split at "_" -> ["btn", "4"] -> take "4" -> make it a number
    face_value = int(ctx.triggered_id.split("_")[1])

    # --- Step 2: Clean up the text the user typed ---
    # (value can be None, so "or ''" turns None into an empty string;
    #  strip() removes accidental spaces at the start/end)
    last_name = (last_name or "").strip()
    first_name = (first_name or "").strip()
    die_name = (die_name or "").strip()

    # --- Step 3: Check the inputs, then save the roll if everything is OK ---
    if not (last_name and first_name and die_name):
        error = "Please fill out all fields."

    elif entry_count >= MAX_ENTRIES:
        error = f"Maximum {MAX_ENTRIES} entries reached. Please stop the app."
        terminate = "App will not accept more entries. Close the browser tab or stop the script."

    else:
        save_roll(last_name, first_name, die_name, face_value)
        entry_count += 1
        status = f"Roll submitted! Total entries: {entry_count}. (Min 10 rolls recommended)"

        if entry_count >= MAX_ENTRIES:
            terminate = (f"Reached {MAX_ENTRIES} entries. No more submissions allowed. "
                         "Stop the app and send the CSV file.")

    # --- Step 4: Build the chart from whatever data is saved for this die ---
    df = load_data(die_name)
    if df.empty:
        fig = make_empty_figure()
    else:
        fig = make_histogram(df, die_name)

    return fig, error, status, terminate


# ============================================================
# PART 6: START THE APP
# ============================================================
# This "if" means: only start the app when you run THIS file directly
# (not when another file imports it).
if __name__ == "__main__":
    app.run(debug=False)

<IPython.core.display.Javascript object>